# Importation des packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

import requests
from bs4 import BeautifulSoup
import os
import s3fs

In [2]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_KEY_S3 = '/movies_metadata.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_movies = pd.read_csv(file_in,sep=',', header=0)

/tmp/ipykernel_74308/2309787388.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_movies = pd.read_csv(file_in,sep=',', header=0)


In [ ]:
FILE_KEY_S3 = '/credits.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_movies = pd.read_csv(file_in,sep=',', header=0)

adult : on enlève
belongs to collection : transformer en booléen + numéro du film/nombre de films précédents + note moyenne du film précédent
budget : vide si 0, à compléter
genres : décomposer en genre_1, genre_2 etc
homepage : on enlève
id et imdb_id : à conserver pour l'instant puis enlever
original_language : voir la distribution pour éventuellement regrouper
original_title : garder
overview : on enlève
popularity : on enlève
poster_path : on enlève
production_countries et production_companies : décomposer puis voir la distribution
release_date : RAS
revenue : vide si 0, à compléter
runtime : à compléter
spoken_languages : on enlève
status : on filtre sur Released
tagline : on enlève
title : on garde
video : on filtre sur False

pondération de l'erreur avec vote_count
classification non supervisée dans les stats descriptives
ajouter le réalisateur + récompenses
ajouter acteurs + récompenses

In [98]:
data_movies_df = data_movies[data_movies['video'] == False]
data_movies_df = data_movies_df[data_movies_df['status'] == 'Released']
data_movies_df = data_movies_df.drop(columns=['adult', 'homepage', 'overview', 'popularity', 'poster_path', 'spoken_languages', 'tagline', 'status', 'video'])
data_movies_df = data_movies_df.dropna(subset= ['release_date'])
data_movies_df = data_movies_df.dropna(subset= ['imdb_id'])
data_movies_df = data_movies_df.dropna(subset= ['original_language'])

In [77]:
data_movies_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44825 entries, 0 to 45465
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   belongs_to_collection  4460 non-null   object 
 1   budget                 44825 non-null  object 
 2   genres                 44825 non-null  object 
 3   id                     44825 non-null  object 
 4   imdb_id                44825 non-null  object 
 5   original_language      44825 non-null  object 
 6   original_title         44825 non-null  object 
 7   production_companies   44825 non-null  object 
 8   production_countries   44825 non-null  object 
 9   release_date           44825 non-null  object 
 10  revenue                44825 non-null  float64
 11  runtime                44588 non-null  float64
 12  title                  44825 non-null  object 
 13  vote_average           44825 non-null  float64
 14  vote_count             44825 non-null  float64
dtypes: floa

In [94]:
missing_percentage = data_movies_df.isna().sum()

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

MISSING VALUES :
belongs_to_collection    40365
runtime                    237
dtype: int64


In [ ]:
url_wikipedia_fr = "https://fr.wikipedia.org/wiki/"
url_wikipedia_en = "https://en.wikipedia.org/wiki/"
data_movies_df['url'] = url_wikipedia_fr + data_movies_df.title.str.replace(" ", "_")
data_movies_df['url_film'] = url_wikipedia_fr + data_movies_df.title.str.replace(" ", "_") + "_(film)"
data_movies_df['release_year'] = data_movies_df.release_date.str[:4]
data_movies_df['url_film_date'] = url_wikipedia_fr + data_movies_df.title.str.replace(" ", "_") + "_(film,_" + data_movies_df.release_year + ")"


In [138]:
data_movies_df.head()

,belongs_to_collection,budget,genres,id,imdb_id,original_language,original_title,production_companies,production_countries,release_date,revenue,runtime,title,vote_average,vote_count,url,url_film,release_year,url_film_date
0,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,tt0114709,en,Toy Story,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,Toy Story,7.7,5415.0,https://fr.wikipedia.org/wiki/Toy_Story,https://fr.wikipedia.org/wiki/Toy_Story_(film),1995,"https://fr.wikipedia.org/wiki/Toy_Story_(film,..."
1,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,tt0113497,en,Jumanji,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,Jumanji,6.9,2413.0,https://fr.wikipedia.org/wiki/Jumanji,https://fr.wikipedia.org/wiki/Jumanji_(film),1995,"https://fr.wikipedia.org/wiki/Jumanji_(film,_1..."
2,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,tt0113228,en,Grumpier Old Men,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,Grumpier Old Men,6.5,92.0,https://fr.wikipedia.org/wiki/Grumpier_Old_Men,https://fr.wikipedia.org/wiki/Grumpier_Old_Men...,1995,https://fr.wikipedia.org/wiki/Grumpier_Old_Men...
3,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,tt0114885,en,Waiting to Exhale,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,81452156.0,127.0,Waiting to Exhale,6.1,34.0,https://fr.wikipedia.org/wiki/Waiting_to_Exhale,https://fr.wikipedia.org/wiki/Waiting_to_Exhal...,1995,https://fr.wikipedia.org/wiki/Waiting_to_Exhal...
4,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",11862,tt0113041,en,Father of the Bride Part II,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,76578911.0,106.0,Father of the Bride Part II,5.7,173.0,https://fr.wikipedia.org/wiki/Father_of_the_Br...,https://fr.wikipedia.org/wiki/Father_of_the_Br...,1995,https://fr.wikipedia.org/wiki/Father_of_the_Br...


In [128]:
data=data_movies_df

In [150]:
i=1

url_film = data_movies_df.url[i]
print(url_film)
liste_acteurs=[]
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
}


r_url_film = requests.get(url_film, headers=headers)


soup_film = BeautifulSoup(r_url_film.text, "html.parser")

selector_homonymie = "div:nth-child(2) > p > a"
homonymie = soup_film.select(selector_homonymie)
homonymie.text

https://fr.wikipedia.org/wiki/Jumanji


AttributeError: ResultSet object has no attribute "text". You're probably treating a list of elements like a single element. Did you call find_all() when you meant to call find()?

In [148]:

if r_url_film.status_code == 404:
    print("url simple ne fonctionne pas")
    liste_acteurs.append("impossible de récupérer la liste des acteurs")
else:
    print("pas erreur 404")
    soup_film = BeautifulSoup(r_url_film.text, "html.parser")

    selector_homonymie = "div:nth-child(2) > p > a"
    homonymie = soup_film.select(selector_homonymie)

    if homonymie[0].text=='page d’homonymie':
        print("homonymie")
        url_film = data_movies_df.url_film_date[i]
        
        r_url_film = requests.get(url_film)

        if r_url_film.status_code == 404:
            print("url avec film et annee ne fonctionne pas")
            url_film = data_movies_df.url_film[i]
            r_url_film = requests.get(url_film)

            if r_url_film.status_code == 404:
                print("url avec annee ne fonctionne pas")
                liste_acteurs.append("impossible de récupérer la liste des acteurs")
            else:
                soup_film = BeautifulSoup(r_url_film.text, "html.parser")

                selector_film = "div.infobox_v3 table tr td div p"
                acteurs = soup_film.select(selector_film)
        
                liste_acteurs.append(acteurs[0].text)
    else:
        print("pas d'homonymie")
        soup_film = BeautifulSoup(r_url_film.text, "html.parser")

        selector_film = "div.infobox_v3 table tr td div p"
        acteurs = soup_film.select(selector_film)
        if len(acteurs)==0:
            print("essai d'autres url")
            url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film,_" + data.release_date[i][0:4] + ")"
        
            r_url_film = requests.get(url_film)

            if r_url_film.status_code == 404:

                url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film)"
                r_url_film = requests.get(url_film)

                if r_url_film.status_code == 404:

                    liste_acteurs.append("impossible de récupérer la liste des acteurs")
                else:
                    soup_film = BeautifulSoup(r_url_film.text, "html.parser")

                    selector_film = "div.infobox_v3 table tr td div p"
                    acteurs = soup_film.select(selector_film)
        
                    liste_acteurs.append(acteurs[0].text)
        else:
            liste_acteurs.append(acteurs[0].text)

pas erreur 404
homonymie


In [183]:
url_film

'https://fr.wikipedia.org/wiki/Balto_(film)'

In [135]:
url_wikipedia = "https://fr.wikipedia.org/wiki/"
liste_acteurs=[]
for i in range(100):
    print(i)
    url_film = url_wikipedia + data.title[i].replace(" ", "_")

    r_url_film = requests.get(url_film, headers = headers)
    if r_url_film.status_code == 404:
        liste_acteurs.append("impossible de récupérer la liste des acteurs")
    else:
        soup_film = BeautifulSoup(r_url_film.text, "html.parser")

        selector_film = "div div p a"
        acteurs = soup_film.select(selector_film)

        if acteurs[0].text=='page d’homonymie':
            url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film,_" + data.release_date[i][0:4] + ")"
        
            r_url_film = requests.get(url_film, headers = headers)

            if r_url_film.status_code == 404:

                url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film)"
                r_url_film = requests.get(url_film, headers = headers)

                if r_url_film.status_code == 404:

                    liste_acteurs.append("impossible de récupérer la liste des acteurs")
                else:
                    soup_film = BeautifulSoup(r_url_film.text, "html.parser")

                    selector_film = "div.infobox_v3 table tr td div p"
                    acteurs = soup_film.select(selector_film)
        
                    liste_acteurs.append(acteurs[0].text)
        else:
            soup_film = BeautifulSoup(r_url_film.text, "html.parser")

            selector_film = "div.infobox_v3 table tr td div p"
            acteurs = soup_film.select(selector_film)
            if len(acteurs)==0:
                print("essai d'autres url")
                url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film,_" + data.release_date[i][0:4] + ")"
        
                r_url_film = requests.get(url_film, headers = headers)

                if r_url_film.status_code == 404:

                    url_film = url_wikipedia + data.title[i].replace(" ", "_") + "_(film)"
                    r_url_film = requests.get(url_film, headers = headers)

                    if r_url_film.status_code == 404:

                        liste_acteurs.append("impossible de récupérer la liste des acteurs")
                    else:
                        soup_film = BeautifulSoup(r_url_film.text, "html.parser")

                        selector_film = "div.infobox_v3 table tr td div p"
                        acteurs = soup_film.select(selector_film)
        
                        liste_acteurs.append(acteurs[0].text)
            else:
                liste_acteurs.append(acteurs[0].text)

0


1
2
3
4
5
6
7
8
9
10
11
12
essai d'autres url
13
14
essai d'autres url
15
16
essai d'autres url
17
18
19
20
21
22
essai d'autres url
23
24
25
26
27
essai d'autres url
28
29
30
31
32
33
essai d'autres url
34
35
36
37
38
39
essai d'autres url
40
essai d'autres url
41
essai d'autres url
42


IndexError: list index out of range

In [136]:
liste_acteurs

['Tom HanksTim Allen\n',
 'impossible de récupérer la liste des acteurs',
 'Whitney Houston Angela BassettLoretta Devine  Lela Rochon\n',
 'impossible de récupérer la liste des acteurs',
 'impossible de récupérer la liste des acteurs',
 'impossible de récupérer la liste des acteurs',
 'Pierce BrosnanSean BeanIzabella ScorupcoFamke JanssenAlan Cumming\n',
 'Michael DouglasAnnette BeningMartin SheenMichael J. Fox\n',
 'impossible de récupérer la liste des acteurs',
 'Kevin BaconBob HoskinsBridget FondaJim CummingsPhil Collins\n',
 'Anthony HopkinsJoan AllenPowers BootheEd Harris\n',
 'Geena DavisMatthew Modine Frank LangellaMaury ChaykinPatrick Malahide\n',
 'Robert De NiroJoe PesciSharon Stone\n',
 'impossible de récupérer la liste des acteurs',
 'Tim Roth  Madonna  Valeria Golino  Jennifer Beals  Antonio Banderas\n',
 'Jim CarreyIan McNeiceSimon CallowBob GuntonMaynard Eziashi\n',
 'Wesley SnipesWoody HarrelsonJennifer LopezRobert Blake\n',
 'John TravoltaGene HackmanRene RussoDanny De